# E3 — Entrenamiento PoinTr v1

**Modelo:** PoinTr (Yu et al., 2021) — transformer para shape completion.
Compite con PCN v4 que está entrenando en paralelo.

**Por qué PoinTr vs PCN:**
- PCN usa un encoder PointNet + folding local → bueno en formas convexas
- PoinTr usa atención global entre puntos visibles y regiones faltantes → capta mejor
  la geometría global de vasijas (simetría axial, huecos internos)
- PoinTr es estado del arte en ShapeNet55 y KITTI; ~40M params vs ~4M de PCN

**Datos:** mismos que PCN v4 — `sintetico_roturas_v2` (2299 pares) + Fantastic Breaks

**Métricas de referencia (PCN v3):** CD=0.0665, F-Score=0.024

⏱️ **Tiempo estimado en A100: ~1-1.5 horas** (300 épocas, batch=32)

⚠️ **CUDA importante:** La Celda 2 compila extensiones CUDA de PoinTr (~10 min). Si falla,
hay un fallback en PyTorch puro (más lento pero funcional).

---
### Antes de ejecutar:
Menú → **Entorno de ejecución → Cambiar tipo → A100 GPU**

In [ ]:
# ── CELDA 1: Montar Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado.')

In [ ]:
# ── CELDA 2: Clonar repos + instalar + compilar CUDA ──────────
#
# Esta celda tarda ~10-15 minutos la primera vez (compilación CUDA).
# Si se interrumpe y se relanza, la compilación salta porque pip detecta
# que el paquete ya está instalado.

import os
import subprocess
import time
from getpass import getpass

# ── 1. Clonar PoinTr oficial ──────────────────────────────────
if not os.path.exists('/content/PoinTr'):
    print('Clonando PoinTr...')
    r = subprocess.run(
        ['git', 'clone', 'https://github.com/yuxumin/PoinTr',
         '/content/PoinTr', '--depth=1', '-q'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('ERROR al clonar PoinTr:', r.stderr)
    else:
        print('PoinTr clonado.')
else:
    print('[OK] /content/PoinTr ya existe.')

# ── 2. Clonar repo TFM ───────────────────────────────────────
REPO_DIR = '/content/TFM'
if not os.path.exists(REPO_DIR):
    token = getpass('Token GitHub (ghp_...): ')
    repo_url = f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D'
    subprocess.run(['git', 'clone', repo_url, REPO_DIR, '-q'],
                   capture_output=True, text=True)
    del token
    print('Repo TFM clonado.')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'],
                   capture_output=True)
    print('[OK] /content/TFM ya existe — actualizado.')

os.chdir(REPO_DIR)
subprocess.run(['git', 'checkout', 'raquel/e3', '-q'], capture_output=True)
print(f'Directorio: {os.getcwd()}')

# ── 3. Instalar dependencias Python ──────────────────────────
print('\nInstalando dependencias Python...')
subprocess.run(['pip', 'install', 'timm', 'easydict', 'pyyaml', '--quiet'])
print('[OK] timm, easydict, pyyaml')

# ── 4. Compilar extensiones CUDA de PoinTr ───────────────────
# pointnet2_ops: KNN + ball query en CUDA (necesario para el encoder DGCNN)
# chamfer_dist:  Chamfer Distance en CUDA (~3x más rápido que cdist)

CUDA_OK = True

for ext_name, ext_path in [
    ('pointnet2_ops', '/content/PoinTr/extensions/pointnet2_ops_lib'),
    ('chamfer_dist',  '/content/PoinTr/extensions/chamfer_dist'),
]:
    print(f'\nCompilando {ext_name}...', flush=True)
    t0 = time.time()
    r = subprocess.run(
        ['pip', 'install', '-e', ext_path, '--quiet'],
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    if r.returncode == 0:
        print(f'  [OK] {ext_name} compilado en {elapsed:.0f}s')
    else:
        print(f'  [WARN] {ext_name} falló ({elapsed:.0f}s) — se usará fallback PyTorch')
        print(f'  Error: {r.stderr[-500:]}')
        CUDA_OK = False

print(f'\nResumen: CUDA extensions = {"OK" if CUDA_OK else "fallback PyTorch"}')
print('Celda 2 completa.')

In [ ]:
# ── CELDA 3: RUTAS DE DRIVE ────────────────────────────────────
DRIVE   = '/content/drive/MyDrive'
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3/Raquel'

VERSION = 'v1_pointr'

RUTA_SINTETICO    = f'{DRIVE}/Datos_E2_E3/General/sintetico_roturas_v2'
RUTA_FB_PROCESADO = f'{DRIVE}/Datos_E2_E3/General/Fantastik_Break_Preprocesado'

# Referencia PCN v4 (para comparar al final)
RUTA_MODELO_PCN_V4 = f'{BASE_E3}/modelos/v4_pcn/best.pt'

RUTA_SALIDA_MODELO     = f'{BASE_E3}/modelos/{VERSION}'
RUTA_SALIDA_RESULTADOS = f'{BASE_E3}/resultados/{VERSION}'

print('Rutas PoinTr v1:')
print(f'  sintetico_v2    : {RUTA_SINTETICO}')
print(f'  fantastic_breaks: {RUTA_FB_PROCESADO}')
print(f'  PCN v4 (ref)    : {RUTA_MODELO_PCN_V4}')
print(f'  salida modelo   : {RUTA_SALIDA_MODELO}')
print(f'  salida resultados: {RUTA_SALIDA_RESULTADOS}')

In [ ]:
# ── CELDA 4: Verificar rutas ────────────────────────────────────
from pathlib import Path

rutas = {
    'sintetico_roturas_v2'   : RUTA_SINTETICO,
    'fantastic_breaks'       : RUTA_FB_PROCESADO,
    'PCN v4 (referencia)'    : RUTA_MODELO_PCN_V4,
}

for nombre, ruta in rutas.items():
    existe = Path(ruta).exists()
    emoji  = 'OK' if existe else 'FALTA'
    print(f'  [{emoji}] {nombre}: {ruta}')

n_sint = len(list(Path(RUTA_SINTETICO).glob('*_completo.npy'))) if Path(RUTA_SINTETICO).exists() else 0
n_fb   = len(list(Path(RUTA_FB_PROCESADO).glob('*_completo.npy'))) if Path(RUTA_FB_PROCESADO).exists() else 0
print(f'\nPares disponibles:')
print(f'  sintetico_v2    : {n_sint}')
print(f'  fantastic_breaks: {n_fb}')
print(f'  TOTAL           : {n_sint + n_fb}')

In [ ]:
# ── CELDA 5: Copiar datos desde Drive ──────────────────────────
import subprocess
from pathlib import Path

def copiar_dir(src, dst):
    src, dst = Path(src), Path(dst)
    if dst.exists() and any(dst.glob('*.npy')):
        n = len(list(dst.glob('*.npy')))
        print(f'  [OK] ya existe: {dst.name}  ({n} .npy)')
        return
    if not src.exists():
        print(f'  [ERROR] no encontrado: {src}')
        return
    dst.mkdir(parents=True, exist_ok=True)
    print(f'  Copiando {src.name}...', flush=True)
    r = subprocess.run(['rsync', '-a', '--no-links', f'{src}/', str(dst)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  [ERROR rsync] {r.stderr[:300]}')
    else:
        n = len(list(dst.glob('*.npy')))
        print(f'  listo — {n} archivos .npy copiados.')

copiar_dir(RUTA_SINTETICO,    'Datos/sintetico/roturas_v2')
copiar_dir(RUTA_FB_PROCESADO, 'Datos/fantastic_breaks/procesado')

print()
for c in ['Datos/sintetico/roturas_v2', 'Datos/fantastic_breaks/procesado']:
    n = len(list(Path(c).glob('*.npy'))) if Path(c).exists() else 0
    print(f'  {c}: {n} .npy')

In [ ]:
# ── CELDA 6: Verificar GPU ─────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'PyTorch CUDA: {torch.version.cuda}')
else:
    print('Sin GPU. Ve a Entorno de ejecución → Cambiar tipo → A100 o T4')

In [ ]:
# ── CELDA 7: ENTRENAR PoinTr v1 ────────────────────────────────
#
# Hiperparámetros clave:
#   num_pred=2048  — igual que PCN (contrato E2→E3)
#   num_query=128  — tokens que representan la región faltante (ratio=16=4^2, limpio)
#   trans_dim=384  — dimensión del transformer (paper original)
#   batch_size=32  — PoinTr pesa ~10x más que PCN en VRAM
#   epochs=300     — PoinTr converge más rápido que PCN por atención global
#   AdamW + CosineAnnealingLR — scheduler estándar para transformers

import sys, os, math, time
from pathlib import Path

sys.path.insert(0, '/content/PoinTr')
sys.path.insert(0, '/content/TFM')
os.chdir('/content/TFM')

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from easydict import EasyDict

# ── Chamfer distance ─────────────────────────────────────────
# Usa la implementación CUDA de PoinTr si se compiló.
# Si no, usa cdist (PyTorch puro, más lento pero idéntico resultado).

try:
    from extensions.chamfer_dist import ChamferDistanceL1
    _cf = ChamferDistanceL1()
    def chamfer_distance(pred, gt):
        return _cf(pred.contiguous(), gt.contiguous())
    print('[OK] Chamfer CUDA de PoinTr')
except Exception as e:
    print(f'[fallback] Chamfer PyTorch puro (razon: {e})')
    def chamfer_distance(pred, gt):
        dist = torch.cdist(pred, gt, p=2)
        return (dist.min(dim=2).values.mean() + dist.min(dim=1).values.mean()) / 2

# ── Subsample GT para loss coarse ────────────────────────────
def subsample_gt(gt, n):
    """Submuestrea aleatoriamente gt a n puntos para supervisar la salida coarse."""
    idx = torch.randperm(gt.size(1), device=gt.device)[:n]
    return gt[:, idx, :]

# ── Modelo PoinTr ─────────────────────────────────────────────
# Intentamos importar el modelo oficial. Si falla por incompatibilidad
# de extensiones, lo sabemos aquí y ajustamos.

import yaml

# Leemos la config base de PoinTr para PCN benchmark
cfg_path = '/content/PoinTr/cfgs/PCN_models/PoinTr.yaml'
if not os.path.exists(cfg_path):
    # Si la ruta cambió en versiones nuevas del repo, buscamos
    import glob as _glob
    candidates = _glob.glob('/content/PoinTr/cfgs/**/PoinTr.yaml', recursive=True)
    cfg_path = candidates[0] if candidates else None

if cfg_path and os.path.exists(cfg_path):
    with open(cfg_path) as f:
        raw_cfg = yaml.safe_load(f)
    model_cfg = EasyDict(raw_cfg.get('model', raw_cfg))
    print(f'Config base leída desde: {cfg_path}')
    print(f'  num_pred original: {model_cfg.get("num_pred")}')
    print(f'  num_query original: {model_cfg.get("num_query")}')
else:
    print('WARN: no se encontró PoinTr.yaml, construyendo config manualmente')
    model_cfg = EasyDict({'NAME': 'PoinTr'})

# Ajustamos para nuestro formato: 2048 puntos de salida
# Ratio 2048/128 = 16 = 4^2 → fold_step=4, cada query genera 4x4=16 puntos
model_cfg.num_pred  = 2048
model_cfg.num_query = 128
# trans_dim y el resto del config se hereda del YAML de PoinTr

print(f'Config ajustada: num_pred={model_cfg.num_pred}, num_query={model_cfg.num_query}')

# Cargar modelo
try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(model_cfg)
    print('[OK] Modelo cargado via build_model_from_cfg')
except Exception as e1:
    print(f'build_model_from_cfg falló: {e1}')
    try:
        from models.PoinTr import PoinTr
        model = PoinTr(model_cfg)
        print('[OK] Modelo cargado via PoinTr directa')
    except Exception as e2:
        raise RuntimeError(
            f'No se pudo inicializar PoinTr.\n'
            f'Error 1: {e1}\nError 2: {e2}\n'
            f'Revisa que la celda 2 compiló las extensiones CUDA correctamente.'
        )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametros entrenables: {n_params:,}')
print(f'Dispositivo: {device}')

# ── Dataset ──────────────────────────────────────────────────
from E3.dataset import construir_dataloaders

CARPETAS   = ['Datos/sintetico/roturas_v2', 'Datos/fantastic_breaks/procesado']
BATCH_SIZE = 32   # PoinTr es ~10x mas pesado en VRAM que PCN

train_loader, val_loader, test_loader = construir_dataloaders(
    carpetas=CARPETAS,
    batch_size=BATCH_SIZE,
)
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

# ── Hiper-parametros de entrenamiento ────────────────────────
EPOCHS   = 300
LR       = 1e-4
W_COARSE = 0.5    # peso de la perdida sobre la salida coarse (proxy points)

CKPT_DIR = Path('E3/checkpoints_pointr')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

mejor_val   = math.inf
train_losses, val_losses = [], []

# ── Bucle de entrenamiento ───────────────────────────────────
print(f'\n{"Epoca":>7}  {"Train":>10}  {"Val":>10}  {"LR":>9}  {"Tiempo":>7}')
print('-' * 52)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── TRAIN ────────────────────────────────────────────────
    model.train()
    total_train = 0.0
    for roto, completo in train_loader:
        roto, completo = roto.to(device), completo.to(device)
        optimizer.zero_grad()

        out = model(roto)                     # tuple: (coarse, fine) o lista
        coarse = out[0] if isinstance(out, (list, tuple)) else out
        fine   = out[-1] if isinstance(out, (list, tuple)) else out

        gt_coarse = subsample_gt(completo, coarse.size(1))
        loss = (chamfer_distance(fine, completo)
                + W_COARSE * chamfer_distance(coarse, gt_coarse))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        total_train += loss.item()

    # ── VAL ──────────────────────────────────────────────────
    model.eval()
    total_val = 0.0
    with torch.no_grad():
        for roto, completo in val_loader:
            roto, completo = roto.to(device), completo.to(device)
            out = model(roto)
            coarse = out[0] if isinstance(out, (list, tuple)) else out
            fine   = out[-1] if isinstance(out, (list, tuple)) else out
            gt_coarse = subsample_gt(completo, coarse.size(1))
            total_val += (chamfer_distance(fine, completo)
                         + W_COARSE * chamfer_distance(coarse, gt_coarse)).item()

    loss_train = total_train / len(train_loader)
    loss_val   = total_val   / len(val_loader)
    elapsed    = time.time() - t0

    train_losses.append(loss_train)
    val_losses.append(loss_val)
    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]

    print(f'{epoch:>4}/{EPOCHS}  {loss_train:>10.6f}  {loss_val:>10.6f}'
          f'  {lr_now:>9.2e}  {elapsed:>6.1f}s')

    # ── checkpoint cada 10 epocas ────────────────────────────
    if epoch % 10 == 0:
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_losses': train_losses, 'val_losses': val_losses,
            'model_cfg': dict(model_cfg),
        }, CKPT_DIR / f'epoch_{epoch:03d}.pt')

    # ── mejor modelo ─────────────────────────────────────────
    if loss_val < mejor_val:
        mejor_val = loss_val
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_losses': train_losses, 'val_losses': val_losses,
            'model_cfg': dict(model_cfg),
        }, CKPT_DIR / 'best.pt')
        print(f'  -> best.pt guardado (val={mejor_val:.6f})')

print(f'\nFin. Mejor val loss: {mejor_val:.6f}')
print(f'Checkpoints en: {CKPT_DIR}')

In [ ]:
# ── CELDA 8: Guardar modelo PoinTr en Drive ────────────────────
import shutil
from pathlib import Path

Path(RUTA_SALIDA_MODELO).mkdir(parents=True, exist_ok=True)

# Guardar best.pt
shutil.copy2('E3/checkpoints_pointr/best.pt', f'{RUTA_SALIDA_MODELO}/best.pt')
print(f'Modelo PoinTr v1 guardado: {RUTA_SALIDA_MODELO}/best.pt')

# Guardar checkpoints periodicos
for ckpt in sorted(Path('E3/checkpoints_pointr').glob('epoch_*.pt')):
    shutil.copy2(ckpt, Path(RUTA_SALIDA_MODELO) / ckpt.name)
    print(f'  + {ckpt.name}')

In [ ]:
# ── CELDA 9: Evaluar PoinTr v1 ─────────────────────────────────
# Mismas metricas que PCN: CD-L1 y F-Score (umbral 0.01)

import torch
import numpy as np
from pathlib import Path
import sys, os

sys.path.insert(0, '/content/PoinTr')
sys.path.insert(0, '/content/TFM')
os.chdir('/content/TFM')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Cargar modelo ────────────────────────────────────────────
ckpt_path = 'E3/checkpoints_pointr/best.pt'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'Cargando best.pt — entrenado hasta epoca {ckpt["epoch"]}')

from easydict import EasyDict
try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(EasyDict(ckpt['model_cfg']))
except Exception:
    from models.PoinTr import PoinTr
    model = PoinTr(EasyDict(ckpt['model_cfg']))

model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device)
model.eval()

# ── Dataset de test ──────────────────────────────────────────
from E3.dataset import construir_dataloaders
_, _, test_loader = construir_dataloaders(
    carpetas=['Datos/sintetico/roturas_v2', 'Datos/fantastic_breaks/procesado'],
    batch_size=32, augmentar=False,
)

# ── Chamfer y F-Score ────────────────────────────────────────
try:
    from extensions.chamfer_dist import ChamferDistanceL1
    _cf = ChamferDistanceL1()
    def chamfer_eval(pred, gt): return _cf(pred.contiguous(), gt.contiguous()).item()
except Exception:
    def chamfer_eval(pred, gt):
        dist = torch.cdist(pred, gt, p=2)
        return ((dist.min(dim=2).values.mean() + dist.min(dim=1).values.mean()) / 2).item()

def fscore(pred, gt, umbral=0.01):
    dist_pg = torch.cdist(pred, gt, p=2)
    dist_gp = torch.cdist(gt, pred, p=2)
    prec = (dist_pg.min(dim=2).values < umbral).float().mean()
    rec  = (dist_gp.min(dim=2).values < umbral).float().mean()
    if prec + rec < 1e-8: return 0.0
    return (2 * prec * rec / (prec + rec)).item()

# ── Evaluacion ───────────────────────────────────────────────
cds, fscores = [], []
with torch.no_grad():
    for roto, completo in test_loader:
        roto, completo = roto.to(device), completo.to(device)
        out = model(roto)
        fine = out[-1] if isinstance(out, (list, tuple)) else out
        for i in range(len(roto)):
            p = fine[i:i+1]
            g = completo[i:i+1]
            cds.append(chamfer_eval(p, g))
            fscores.append(fscore(p, g))

cd_mean = np.mean(cds)
fs_mean = np.mean(fscores)

print(f'\n=== PoinTr v1 — Resultados test ===')
print(f'  CD-L1   : {cd_mean:.4f}')
print(f'  F-Score : {fs_mean:.4f}')
print(f'  Muestras: {len(cds)}')

# Guardar resumen
resumen_dir = Path('E3/resultados_pointr')
resumen_dir.mkdir(parents=True, exist_ok=True)
with open(resumen_dir / 'resumen.txt', 'w') as f:
    f.write(f'Modelo: PoinTr v1\n')
    f.write(f'Epoca best: {ckpt["epoch"]}\n')
    f.write(f'CD-L1: {cd_mean:.6f}\n')
    f.write(f'F-Score: {fs_mean:.6f}\n')
    f.write(f'Muestras test: {len(cds)}\n')

# Guardar en Drive
import shutil
Path(RUTA_SALIDA_RESULTADOS).mkdir(parents=True, exist_ok=True)
shutil.copytree('E3/resultados_pointr', RUTA_SALIDA_RESULTADOS, dirs_exist_ok=True)
print(f'Resultados guardados: {RUTA_SALIDA_RESULTADOS}')

In [ ]:
# ── CELDA 10: Comparar PCN v4 vs PoinTr v1 ─────────────────────
from pathlib import Path

comparaciones = [
    ('PCN v3 (referencia)',  None,                                         '0.066500', '0.024000'),
    ('PCN v4',              f'{BASE_E3}/resultados/v4_pcn/resumen.txt',   None,       None),
    ('PoinTr v1',           'E3/resultados_pointr/resumen.txt',           None,       None),
]

print('=' * 60)
print(f'{"Modelo":<20}  {"CD-L1":>10}  {"F-Score":>10}  {"Mejor que v3"}')
print('=' * 60)

ref_cd = 0.0665

for nombre, ruta, cd_fallback, fs_fallback in comparaciones:
    if ruta is None:
        cd_str, fs_str = cd_fallback, fs_fallback
        cd_val = float(cd_fallback)
    elif Path(ruta).exists():
        txt = Path(ruta).read_text()
        cd_str = fs_str = '?'
        cd_val = None
        for line in txt.splitlines():
            if line.startswith('CD-L1:'):
                cd_str = line.split(':')[1].strip()
                cd_val = float(cd_str)
            if line.startswith('F-Score:'):
                fs_str = line.split(':')[1].strip()
    else:
        cd_str = fs_str = 'no disponible'
        cd_val = None

    mejora = ''
    if cd_val is not None and nombre != 'PCN v3 (referencia)':
        pct = (ref_cd - cd_val) / ref_cd * 100
        mejora = f'{pct:+.1f}%' if abs(pct) > 0.1 else '='

    print(f'{nombre:<20}  {cd_str:>10}  {fs_str:>10}  {mejora}')

print('=' * 60)
print('\nNota: CD-L1 menor = mejor | F-Score mayor = mejor')

In [ ]:
# ── CELDA 11: Curvas de aprendizaje ────────────────────────────
import matplotlib.pyplot as plt
import torch

ckpt = torch.load('E3/checkpoints_pointr/best.pt', map_location='cpu', weights_only=False)
train_losses = ckpt['train_losses']
val_losses   = ckpt['val_losses']
best_epoch   = ckpt['epoch']

fig, ax = plt.subplots(figsize=(10, 4))
epochs = range(1, len(train_losses) + 1)
ax.plot(epochs, train_losses, label='Train', color='#1565C0', alpha=0.8)
ax.plot(epochs, val_losses,   label='Val',   color='#C62828', alpha=0.8)
ax.axvline(best_epoch, color='#2E7D32', linestyle='--', alpha=0.7, label=f'best (e{best_epoch})')
ax.set_xlabel('Epoca')
ax.set_ylabel('Loss (CD-L1 + 0.5*CD-coarse)')
ax.set_title('PoinTr v1 — curvas de aprendizaje')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('E3/resultados_pointr/curvas.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Mejor epoch: {best_epoch} | val loss: {min(val_losses):.6f}')

In [ ]:
# ── CELDA 12: Visualizacion 3D interactiva ─────────────────────
#   Azul  = entrada rota
#   Verde = GT completa
#   Rojo  = prediccion PoinTr

import subprocess
subprocess.run(['pip', 'install', 'plotly', '--quiet'])

import plotly.graph_objects as go
import numpy as np
import torch, sys, os

sys.path.insert(0, '/content/PoinTr')
sys.path.insert(0, '/content/TFM')
os.chdir('/content/TFM')

from easydict import EasyDict
from E3.dataset import construir_dataloaders

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt = torch.load('E3/checkpoints_pointr/best.pt', map_location=device, weights_only=False)
try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(EasyDict(ckpt['model_cfg']))
except Exception:
    from models.PoinTr import PoinTr
    model = PoinTr(EasyDict(ckpt['model_cfg']))
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device)
model.eval()
print(f'Modelo cargado — epoca {ckpt["epoch"]}')

_, _, test_loader = construir_dataloaders(
    carpetas=['Datos/sintetico/roturas_v2', 'Datos/fantastic_breaks/procesado'],
    batch_size=32, augmentar=False,
)

def chamfer_np(a, b):
    a, b = torch.tensor(a).unsqueeze(0), torch.tensor(b).unsqueeze(0)
    dist = torch.cdist(a, b, p=2)
    return ((dist.min(2).values.mean() + dist.min(1).values.mean()) / 2).item()

rotos, gts, preds, cds = [], [], [], []
with torch.no_grad():
    for roto_b, gt_b in test_loader:
        out = model(roto_b.to(device))
        pred_b = (out[-1] if isinstance(out, (list, tuple)) else out).cpu()
        for i in range(len(roto_b)):
            rotos.append(roto_b[i].numpy())
            gts.append(gt_b[i].numpy())
            preds.append(pred_b[i].numpy())
            cds.append(chamfer_np(pred_b[i].numpy(), gt_b[i].numpy()))

cds_arr = np.array(cds)
orden   = np.argsort(cds_arr)
indices = list(orden[:3]) + list(orden[-3:])
titulos = ['Mejor 1', 'Mejor 2', 'Mejor 3', 'Peor 1', 'Peor 2', 'Peor 3']
print(f'CD — mejor: {cds_arr[orden[0]]:.4f} | peor: {cds_arr[orden[-1]]:.4f} | media: {cds_arr.mean():.4f}')

for idx, titulo in zip(indices, titulos):
    fig = go.Figure([
        go.Scatter3d(x=rotos[idx][:,0], y=rotos[idx][:,1], z=rotos[idx][:,2],
                     mode='markers', marker=dict(size=2, color='#4C72B0', opacity=0.55), name='Rota'),
        go.Scatter3d(x=gts[idx][:,0],   y=gts[idx][:,1],   z=gts[idx][:,2],
                     mode='markers', marker=dict(size=2, color='#55A868', opacity=0.35), name='GT'),
        go.Scatter3d(x=preds[idx][:,0], y=preds[idx][:,1], z=preds[idx][:,2],
                     mode='markers', marker=dict(size=2, color='#C44E52', opacity=0.85), name='PoinTr'),
    ])
    fig.update_layout(
        title=f'{titulo} — CD={cds[idx]:.4f}',
        scene=dict(xaxis=dict(range=[-1,1], showticklabels=False),
                   yaxis=dict(range=[-1,1], showticklabels=False),
                   zaxis=dict(range=[-1,1], showticklabels=False),
                   aspectmode='cube'),
        height=500, margin=dict(l=0,r=0,b=0,t=40)
    )
    fig.show()